# Parsing statistics

<!-- Drafted with the help of an AI assistant (Claude, Anthropic), disclosed per course rules. -->

Every number and figure on this page is computed when the book is built from `data/m2/m2_papers.csv` (one row per paper), `data/m2/m2_versions.csv` (one row per version) and `data/m2/structure_audit.csv`; figures are interactive (hover, zoom, click the legend).

:::{warning}
**Scope of these numbers.** Until the full crawl is parsed, the CSV files hold the output of the real M2 pipeline on the **100-paper smoke crawl of M1** (`2606.25734`-`2606.25833`), a small subset. They show how the parser behaves, not the final statistics of the corpus; the files are replaced by the export of the official run without changing this page.
:::

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA = Path("../data/m2")
papers = pd.read_csv(DATA / "m2_papers.csv", dtype={"arxiv_id": str})
versions = pd.read_csv(DATA / "m2_versions.csv", dtype={"arxiv_id": str})
audit = pd.read_csv(DATA / "structure_audit.csv", dtype={"arxiv_id": str})

## Parsing success rate

A version is **parsed** when it has a root `.tex` file and every stage ran. A version **without TeX** (PDF-only, unavailable at arXiv) is recorded as `empty`; `no_root` means `.tex` files but none with `\documentclass`; `error` means a stage failed on it; `missing` means the crawl produced no folder. The *success rate* is computed over the versions that have a TeX source; *coverage* is over all versions of the corpus.

In [ ]:
counts = versions["status"].value_counts()
with_tex = counts.get("parsed", 0) + counts.get("no_root", 0) + counts.get("error", 0)
summary = pd.Series(
    {
        "Papers": len(papers),
        "Versions": len(versions),
        "Parsed": int(counts.get("parsed", 0)),
        "Empty (no TeX, recorded)": int(counts.get("empty", 0)),
        "No root file": int(counts.get("no_root", 0)),
        "Errors": int(counts.get("error", 0)),
        "Missing folder": int(counts.get("missing", 0)),
        "Success rate over versions with TeX (%)": round(
            100 * counts.get("parsed", 0) / with_tex, 2
        ) if with_tex else float("nan"),
        "Coverage over all versions (%)": round(100 * counts.get("parsed", 0) / len(versions), 2),
    },
    dtype=object,
).to_frame("Value")
summary

In [ ]:
fig = px.pie(
    counts.rename_axis("status").reset_index(name="versions"),
    names="status",
    values="versions",
    title="Outcome of every version",
    hole=0.4,
)
fig

In [ ]:
reasons = (
    versions[versions["status"] != "parsed"]
    .groupby(["status", "reason"], dropna=False)
    .size()
    .rename("versions")
    .reset_index()
    .sort_values("versions", ascending=False)
)
reasons

## Version coverage per publication

In [ ]:
papers["share_parsed"] = papers["n_parsed"] / papers["n_versions"]
fig = px.histogram(
    papers,
    x="n_versions",
    color=papers["share_parsed"].map(
        lambda s: "all versions parsed" if s == 1 else ("none parsed" if s == 0 else "some parsed")
    ),
    labels={"n_versions": "Versions of the publication", "color": "Coverage"},
    title="Versions per publication, by how many were parsed",
)
fig.update_layout(bargap=0.1, yaxis_title="Publications")
fig

## Elements and references before and after deduplication

*Before* is what the trees of all versions hold (every sentence, formula, figure and heading of every version, the document roots excluded; every reference entry of every version). *After* is what the tables store: equal elements of different versions share one row, and references that are the same work are united.

In [ ]:
totals = pd.DataFrame(
    {
        "Before deduplication": [papers["elements_before"].sum(), papers["bibtex_before"].sum()],
        "After deduplication": [
            papers["elements"].sum() - versions["status"].ne("missing").sum(),
            papers["bibtex"].sum(),
        ],
    },
    index=["Elements (roots excluded)", "Bibliography entries"],
)
totals["Reduction (%)"] = (100 * (1 - totals.iloc[:, 1] / totals.iloc[:, 0])).round(1)
totals

In [ ]:
multi = papers[papers["n_parsed"] > 1].copy()
multi["roots"] = multi["n_versions"] - multi["n_missing"]
multi["kept"] = (multi["elements"] - multi["roots"]) / multi["elements_before"]
fig = px.histogram(
    multi,
    x="kept",
    nbins=30,
    labels={"kept": "Elements stored / elements found (1 = nothing shared)"},
    title=f"Papers with several parsed versions (n = {len(multi)}): share of elements stored",
)
fig.update_layout(yaxis_title="Publications")
fig

## References

In [ ]:
parsed = versions[versions["status"] == "parsed"]
no_refs = parsed["n_refs"].fillna(0).eq(0)
refs_summary = pd.Series(
    {
        "Parsed versions": len(parsed),
        "... with at least one reference": int((~no_refs).sum()),
        "... without any reference (no .bib, \\bibitem or .bbl in the corpus)": int(no_refs.sum()),
        "Versions whose references came from a .bbl": int(parsed["n_bbl"].fillna(0).gt(0).sum()),
        "Versions that name a .bib the corpus does not have": int(parsed["missing_bib"].notna().sum()),
        "Reference entries (before deduplication)": int(parsed["n_refs"].fillna(0).sum()),
    },
    dtype=object,
).to_frame("Value")
refs_summary

## Does the tree keep what the source has?

For each parsed version the source is read again and four things are counted with plain regular expressions (headings, displayed formulas, floats, list items); the table gives the share of versions where the count in the tree is equal. A difference points to a structure that was lost or invented. Identical formulas under the same parent are stored once by design, which explains most of the formula differences.

In [ ]:
rates = pd.DataFrame(
    {
        "Versions where source and tree agree": [int(audit[f"{m}_agree"].sum()) for m in
            ["headings", "formulas", "floats", "items"]],
        "Versions compared": len(audit),
    },
    index=["Headings", "Displayed formulas", "Floats (figures and tables)", "List items"],
)
rates["Agreement (%)"] = (100 * rates.iloc[:, 0] / rates.iloc[:, 1]).round(1)
rates